In [11]:
import import_ipynb
from chapter23_4Node_Quad import Quad4IsoPMembraneStiffness
import numpy as np


def MembraneSolution(nodcoor, eletyp, elenod, elemat, elefab, eleopt, doftag, dofval):
    """
    INPUT: nodcoor (list of coords [[x1,y1], [x2, y2], etc.])
           eletyp (list of element type [[ etyp ], [ etyp ], etc])
           elenod (connectivity, list of nodes for each element)
           elemat (list with E, ν , for each element [[E, ν], etc]
           elefab (list wiht plate thickness per element)
           eleopt (kept for faithfulness to Felippa, specifies float number in Mathematica)
           doftag (freedom tags, like nodcoor but 0 means force is specified, else 1. [[0,0], [1,0], etc])
           dofval (freedom values, lists either force or prescribed displacements, depends on doftag)

    OUTPUT: u (displacement vector)
            f (force vector)
            sig (stresses)
    """
    numnod = len(nodcoor)
    numele = len(elenod)

    K = MembraneMasterStiffness(nodcoor, eletyp, elenod, elemat, elefab, eleopt)

    ns = 0
    for n in range(numnod):
        for j in range(2):
            if doftag[n][j] > 0:
                ns += 1

    supdof = np.zeros(ns, dtype=int)  
    supval = np.zeros(ns)              

    k = 0
    for n in range(numnod):
        for j in range(2):
            if doftag[n][j] > 0:
                supdof[k] = 2 * n + (j + 1)   
                supval[k] = dofval[n][j]
                k += 1

    f = ModifiedNodeForces(supdof, supval, K, np.asarray(dofval).flatten())
    Kmod = ModifiedMasterStiffness(supdof, K)

    u = np.linalg.solve(Kmod, f)
    u = np.where(np.abs(u) < 1e-10, 0, u)

    f = K @ u
    f = np.where(np.abs(f) < 1e-10, 0, f)

    sig = MembraneNodalStresses(nodcoor, eletyp, elenod, elemat, elefab, eleopt, u)
    sig = np.where(np.abs(sig) < 1e-10, 0, sig)

    return u, f, sig


def MembraneMasterStiffness(nodcoor, eletyp, elenod, elemat, elefab, eleopt):
    """
    INPUT: nodcoor (list of coords [[x1,y1], [x2, y2], etc.])
           eletyp (list of element type [[ etyp ], [ etyp ], etc])
           elenod (connectivity, list of nodes for each element)
           elemat (list with E, ν , for each element [[E, ν], etc]
           elefab (list wiht plate thickness per element)
           eleopt (kept for faithfulness to Felippa, specifies float number in Mathematica)

    OUTPUT: K (master stiffness matrix)
    """
    numele = len(elenod)
    numnod = len(nodcoor)

    nodcoor = np.asarray(nodcoor, dtype=float)
    assert nodcoor.ndim == 2 and nodcoor.shape[1] == 2, \
        f"nodcoor must be (numnod, 2), got {nodcoor.shape}"

    K = np.zeros((2 * numnod, 2 * numnod))
    numer = eleopt[0]

    for ne in range(numele):                       
        type_specific = eletyp[ne]                 # in original, variable is named "type"

        if type_specific not in ["Trig3", "Trig6", "Trig10", "Quad4", "Quad9"]:
            print(f"Illegal element type, ne={ne}")
            return None

        eNL = elenod[ne]
        n = len(eNL)

        eftab = []
        for e in eNL:
            eftab += [2 * e - 1, 2 * e]

        ncoor = []
        for e in eNL:
            ncoor.append(nodcoor[e - 1])
            
        Em, nu = elemat[ne]                         
        Emat = Em / (1 - nu ** 2) * np.array([
            [1,  nu, 0],
            [nu, 1,  0],
            [0,  0,  (1 - nu) / 2],
        ])

        th = elefab[ne][0]  
        
        if type_specific == "Trig3":
            Ke = Trig3IsoPMembraneStiffness(ncoor, [Emat, 0, 0], [th], [numer])
        elif type_specific == "Quad4":
            Ke = Quad4IsoPMembraneStiffness(ncoor, [Emat, 0, 0], [th], [numer, 2])
        elif type_specific == "Trig6":
            Ke = Trig6IsoPMembraneStiffness(ncoor, [Emat, 0, 0], [th], [numer, 3])
        elif type_specific == "Quad9":
            Ke = Quad9IsoPMembraneStiffness(ncoor, [Emat, 0, 0], [th], [numer, 3])
        elif type_specific == "Trig10":
            Ke = Trig10IsoPMembraneStiffness(ncoor, [Emat, 0, 0], [th], [numer, 3])

        neldof = len(Ke)

        # assemble upper triangle, then mirror it into lower 
        for i in range(neldof):
            ii = eftab[i]
            for j in range(i, neldof):
                jj = eftab[j]
                K[ii - 1, jj - 1] += Ke[i, j]
                K[jj - 1, ii - 1] = K[ii - 1, jj - 1]

    return K

In [21]:
s = [1, 0.70, 0.48, 0.30, 0.16, 0.07, 0.0]
 
xy1  = np.array([0, 6])
xy7  = np.array([0, 1])
xy8  = np.array([2.5, 6])
xy14 = np.array([np.cos(3 * np.pi / 8), np.sin(3 * np.pi / 8)])
xy21 = np.array([np.cos(np.pi / 4),     np.sin(np.pi / 4)])
xy15 = np.array([5, 6])
xy22 = np.array([5, 2])
xy28 = np.array([np.cos(np.pi / 8),     np.sin(np.pi / 8)])
xy29 = np.array([5, 0])
xy35 = np.array([1, 0])
 
NodeCoordinates = np.zeros((35, 2))
for n in range(1, 8):     # nodes 1..7
    NodeCoordinates[n - 1] = s[n - 1]  * xy1  + (1 - s[n - 1])  * xy7
for n in range(8, 15):    # nodes 8..14
    NodeCoordinates[n - 1] = s[n - 8]  * xy8  + (1 - s[n - 8])  * xy14
for n in range(15, 22):   # nodes 15..21
    NodeCoordinates[n - 1] = s[n - 15] * xy15 + (1 - s[n - 15]) * xy21
for n in range(22, 29):   # nodes 22..28
    NodeCoordinates[n - 1] = s[n - 22] * xy22 + (1 - s[n - 22]) * xy28
for n in range(29, 36):   # nodes 29..35
    NodeCoordinates[n - 1] = s[n - 29] * xy29 + (1 - s[n - 29]) * xy35

ElemNodeLists = np.zeros((24, 4), dtype=int)
ElemNodeLists[0] = [1, 2, 9, 8]
for e in range(2, 7):     # elements 2..6
    ElemNodeLists[e - 1] = ElemNodeLists[e - 2] + 1
ElemNodeLists[6] = ElemNodeLists[5] + 2
for e in range(8, 13):    # elements 8..12
    ElemNodeLists[e - 1] = ElemNodeLists[e - 2] + 1
ElemNodeLists[12] = ElemNodeLists[11] + 2
for e in range(14, 19):   # elements 14..18
    ElemNodeLists[e - 1] = ElemNodeLists[e - 2] + 1
ElemNodeLists[18] = ElemNodeLists[17] + 2
for e in range(20, 25):   # elements 20..24
    ElemNodeLists[e - 1] = ElemNodeLists[e - 2] + 1
 
# convert to a plain list of lists, which is what the module iterates over
ElemNodeLists = ElemNodeLists.tolist()
 

numnod = len(NodeCoordinates)          # 35
numele = len(ElemNodeLists)            # 24
 
eletyp = ["Quad4"] * numele
elemat = [[1000.0, 0.25]] * numele     
elefab = [[1.0]] * numele              
eleopt = [0]                          

K = MembraneMasterStiffness(NodeCoordinates, eletyp, ElemNodeLists,
                            elemat, elefab, eleopt)
 

eigvals = np.linalg.eigvalsh(K)
eigvals = np.where(np.abs(eigvals) < 1e-10, 0.0, eigvals)

print("Eigenvalues of K=")
print(eigvals)
print(eigvals.shape)


Eigenvalues of K=
[   0.            0.            0.          102.3348758   114.75651474
  147.22457448  163.74275506  188.41286988  264.61514045  289.10463428
  322.71646772  354.46460176  398.43763467  461.08368605  508.48651585
  543.52720071  556.03114225  627.06922073  674.79451888  693.43326369
  729.43483653  732.80526128  799.19220538  850.41424037  865.27620543
  899.546232    917.78083006  952.35380869  983.34654337  997.2219604
 1025.03701802 1081.90287077 1122.16765215 1198.27131293 1217.54265727
 1248.34131525 1281.17430236 1332.4748084  1352.38058335 1398.20261061
 1479.72691008 1521.00613326 1580.4324351  1617.49835044 1658.39754438
 1695.46329336 1778.85283608 1854.18975242 1873.45738162 1945.8667904
 1987.68861854 2058.11377913 2099.81461061 2229.0142277  2282.87240852
 2295.75674077 2448.16633604 2655.80332784 2756.32373342 2957.15593237
 3102.63932959 3172.09176929 3321.2071933  3424.02859688 3582.17484396
 3898.85783291 4328.65994653 4478.74060947 5199.45748687 6479